# プロンプト圧縮 統合評価システム v3

## パイプライン

```
プロンプト入力
    ↓
セマンティックキャッシュ検索
    ↓ ヒット → キャッシュ応答を返す
    ↓ ミス
全手法で並列圧縮（TF-IDF / SelectiveContext / LLMLingua / LongLLMLingua / AttentionPruning）
    ↓
各圧縮結果を評価（ROUGE・意味類似度・レイテンシ → 総合スコア）
    ↓
総合スコア最高の圧縮結果を選択
    ↓
LLM呼び出し（最良の圧縮後プロンプトで）
    ↓
結果をキャッシュ保存
```

## パッケージの位置づけ
| 手法 | 実装 | パッケージ |
|------|------|----------|
| TF-IDF | 自前 | scikit-learn |
| SelectiveContext | **本家** | `selective-context` |
| LLMLingua | **本家** | `llmlingua` |
| LongLLMLingua | **本家** | `llmlingua`（同パッケージ内） |
| AttentionPruning | 自前 | transformers |

## セル1: 環境セットアップ

In [ ]:
!apt-get -qq install redis-server
!pip install -q redis sentence-transformers transformers scikit-learn rouge-score
!pip install -q selective-context
!pip install -q llmlingua
!pip install -q bert-score  # オプション
print('✅ インストール完了')

## セル2: インポートと共有モデルの初期化

In [ ]:
import subprocess, time, json, re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import redis
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

# Redis起動
subprocess.Popen(['redis-server'])
time.sleep(1)
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 共有モデル（遅延ロード）
_embedder = None
_tokenizer = None

def get_embedder():
    global _embedder
    if _embedder is None:
        print('[モデルロード] SentenceTransformer...')
        _embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _embedder

def get_tokenizer():
    global _tokenizer
    if _tokenizer is None:
        print('[モデルロード] tokenizer (flan-t5)...')
        _tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
    return _tokenizer

def token_len(text: str) -> int:
    return len(get_tokenizer().encode(text))

print('✅ 初期化完了')

## セル3: 圧縮手法の定義（プラグイン形式）

In [ ]:
# ============================================================
# 圧縮結果データクラス
# ============================================================
@dataclass
class CompressionResult:
    original: str
    compressed: str
    method: str
    original_tokens: int
    compressed_tokens: int
    latency_ms: float
    ratio: float = field(init=False)

    def __post_init__(self):
        self.ratio = 1 - (self.compressed_tokens / max(self.original_tokens, 1))

    def __str__(self):
        return (
            f'  [{self.method}]\n'
            f'    元トークン数    : {self.original_tokens}\n'
            f'    圧縮後トークン数: {self.compressed_tokens}\n'
            f'    圧縮率          : {self.ratio*100:.1f}%\n'
            f'    レイテンシ      : {self.latency_ms:.1f}ms\n'
            f'    圧縮後テキスト  : {self.compressed[:80]}...'
        )


# ============================================================
# 基底クラス
# ============================================================
class BaseCompressor(ABC):
    @property
    @abstractmethod
    def name(self) -> str: pass

    @abstractmethod
    def _compress(self, text: str, ratio: float) -> str: pass

    def compress(self, text: str, ratio: float = 0.5) -> CompressionResult:
        original_tokens = token_len(text)
        start = time.time()
        compressed = self._compress(text, ratio)
        latency_ms = (time.time() - start) * 1000
        return CompressionResult(
            original=text, compressed=compressed, method=self.name,
            original_tokens=original_tokens,
            compressed_tokens=token_len(compressed),
            latency_ms=latency_ms,
        )


# ============================================================
# 手法1: TF-IDF（自前実装・最軽量）
# ============================================================
class TFIDFCompressor(BaseCompressor):
    @property
    def name(self): return 'TF-IDF'

    def _compress(self, text: str, ratio: float) -> str:
        sentences = [s.strip() for s in re.split(r'[。.!?\n]', text) if s.strip()]
        if len(sentences) <= 1:
            words = text.split()
            return ' '.join(words[:max(1, int(len(words) * (1 - ratio)))])
        keep_n = max(1, int(len(sentences) * (1 - ratio)))
        try:
            from sklearn.feature_extraction.text import TfidfVectorizer
            vec = TfidfVectorizer()
            mat = vec.fit_transform(sentences)
            scores = mat.sum(axis=1).A1
            top = sorted(np.argsort(scores)[-keep_n:].tolist())
            return '。'.join([sentences[i] for i in top])
        except Exception:
            return '。'.join(sentences[:keep_n])


# ============================================================
# 手法2: SelectiveContext（本家パッケージ使用）
# https://github.com/liyucheng09/selective_context
# ============================================================
class SelectiveContextCompressor(BaseCompressor):
    def __init__(self, model_type: str = 'gpt2'):
        self.model_type = model_type
        self._sc = None

    def _get_sc(self):
        if self._sc is None:
            print(f'[モデルロード] SelectiveContext ({self.model_type})...')
            from selective_context import SelectiveContext
            self._sc = SelectiveContext(model_type=self.model_type)
        return self._sc

    @property
    def name(self): return 'SelectiveContext'

    def _compress(self, text: str, ratio: float) -> str:
        compressed, _ = self._get_sc()(text, reduce_ratio=ratio)
        return compressed


# ============================================================
# 手法3: LLMLingua（本家パッケージ使用）
# https://github.com/microsoft/LLMLingua
# ============================================================
class LLMLinguaCompressor(BaseCompressor):
    def __init__(
        self,
        model_name: str = 'microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank',
        device_map: str = 'cpu',
    ):
        self.model_name = model_name
        self.device_map = device_map
        self._compressor = None

    def _get_compressor(self):
        if self._compressor is None:
            print(f'[モデルロード] LLMLingua ({self.model_name})...')
            from llmlingua import PromptCompressor
            self._compressor = PromptCompressor(
                model_name=self.model_name,
                use_llmlingua2=True,
                device_map=self.device_map,
            )
        return self._compressor

    @property
    def name(self): return 'LLMLingua'

    def _compress(self, text: str, ratio: float) -> str:
        result = self._get_compressor().compress_prompt(
            text, rate=1 - ratio, force_tokens=['\n']
        )
        return result['compressed_prompt']


# ============================================================
# 手法4: LongLLMLingua（本家パッケージ使用）
# llmlingua パッケージに内包
# ============================================================
class LongLLMLinguaCompressor(BaseCompressor):
    def __init__(
        self,
        model_name: str = 'microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank',
        device_map: str = 'cpu',
        question: str = '',
    ):
        self.model_name = model_name
        self.device_map = device_map
        self.question = question
        self._compressor = None

    def _get_compressor(self):
        if self._compressor is None:
            print(f'[モデルロード] LongLLMLingua ({self.model_name})...')
            from llmlingua import PromptCompressor
            self._compressor = PromptCompressor(
                model_name=self.model_name,
                device_map=self.device_map,
            )
        return self._compressor

    @property
    def name(self): return 'LongLLMLingua'

    def _compress(self, text: str, ratio: float) -> str:
        kwargs = dict(
            rate=1 - ratio,
            force_tokens=['\n'],
            rank_method='longllmlingua',
            context_budget='+100',
            dynamic_context_compression_ratio=0.4,
            reorder_context='sort',
        )
        if self.question:
            kwargs['question'] = self.question
        result = self._get_compressor().compress_prompt(text, **kwargs)
        return result['compressed_prompt']


# ============================================================
# 手法5: AttentionPruning（自前実装）
# ============================================================
class AttentionPruningCompressor(BaseCompressor):
    def __init__(self, model_name: str = 'google/flan-t5-base'):
        self.model_name = model_name
        self._tok = None
        self._mdl = None

    def _get_model(self):
        if self._mdl is None:
            print(f'[モデルロード] AttentionPruning ({self.model_name})...')
            self._tok = AutoTokenizer.from_pretrained(self.model_name)
            self._mdl = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
        return self._tok, self._mdl

    @property
    def name(self): return 'AttentionPruning'

    def _compress(self, text: str, ratio: float) -> str:
        import torch
        tok, mdl = self._get_model()
        inputs = tok(text, return_tensors='pt', truncation=True, max_length=512)
        with torch.no_grad():
            outputs = mdl.encoder(**inputs, output_attentions=True)
        attentions = torch.stack(outputs.attentions)
        avg_attention = attentions.mean(dim=[0, 1, 2])
        token_importance = avg_attention.sum(dim=0).numpy()
        token_ids = inputs['input_ids'][0].tolist()
        n_keep = max(1, int(len(token_ids) * (1 - ratio)))
        top_indices = sorted(np.argsort(token_importance)[-n_keep:].tolist())
        return tok.decode([token_ids[i] for i in top_indices], skip_special_tokens=True)


# ============================================================
# レジストリ（新手法は1行追加するだけで使用可能）
# ============================================================
COMPRESSOR_REGISTRY: dict[str, type[BaseCompressor]] = {
    'tfidf'      : TFIDFCompressor,
    'selective'  : SelectiveContextCompressor,
    'llmlingua'  : LLMLinguaCompressor,
    'longlingua' : LongLLMLinguaCompressor,
    'attention'  : AttentionPruningCompressor,
}

_compressor_instances: dict[str, BaseCompressor] = {}

def get_compressor(name: str) -> BaseCompressor:
    if name not in COMPRESSOR_REGISTRY:
        raise ValueError(f'未知の手法: {name} / 利用可能: {list(COMPRESSOR_REGISTRY.keys())}')
    if name not in _compressor_instances:
        _compressor_instances[name] = COMPRESSOR_REGISTRY[name]()
    return _compressor_instances[name]

print('✅ 圧縮手法定義完了')
print(f'   利用可能: {list(COMPRESSOR_REGISTRY.keys())}')

## セル4: 評価メトリクスの定義

In [ ]:
@dataclass
class EvaluationResult:
    method: str
    compression_ratio: float
    rouge1: float
    rougeL: float
    semantic_sim: float
    latency_ms: float
    bert_score_f1: float = 0.0
    composite_score: float = 0.0
    compression_result: Optional[object] = field(default=None, repr=False)

    def compute_composite(self):
        """
        総合スコア:
          意味類似度 × 0.40
          ROUGE-L    × 0.30
          圧縮率     × 0.20  （多く削減できるほど高い）
          速度スコア × 0.10  （100ms以内=1.0, 1000ms以上=0.0）
        """
        speed_score = max(0.0, min(1.0, 1.0 - (self.latency_ms - 100) / 900))
        self.composite_score = (
            self.semantic_sim      * 0.40 +
            self.rougeL            * 0.30 +
            self.compression_ratio * 0.20 +
            speed_score            * 0.10
        )
        return self

    def summary_line(self) -> str:
        return (
            f'  [{self.method:<16}] '
            f'総合={self.composite_score:.3f} '
            f'意味={self.semantic_sim:.3f} '
            f'ROUGE-L={self.rougeL:.3f} '
            f'圧縮率={self.compression_ratio*100:.1f}% '
            f'{self.latency_ms:.0f}ms'
        )


class CompressionEvaluator:
    def __init__(self, use_bert_score: bool = False, lang: str = 'ja'):
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
        self.use_bert_score = use_bert_score
        self.lang = lang

    def evaluate(self, result: CompressionResult) -> EvaluationResult:
        # ROUGE
        rs = self.rouge.score(result.original, result.compressed)
        r1 = rs['rouge1'].fmeasure
        rL = rs['rougeL'].fmeasure

        # 意味的類似度
        emb = get_embedder()
        o = emb.encode(result.original)
        c = emb.encode(result.compressed)
        sim = float(np.dot(o, c) / (np.linalg.norm(o) * np.linalg.norm(c) + 1e-10))

        eval_result = EvaluationResult(
            method=result.method,
            compression_ratio=result.ratio,
            rouge1=r1, rougeL=rL,
            semantic_sim=sim,
            latency_ms=result.latency_ms,
            compression_result=result,
        )

        # BERTScore（オプション）
        if self.use_bert_score:
            try:
                from bert_score import score as bs
                _, _, F1 = bs([result.compressed], [result.original],
                              lang=self.lang, verbose=False)
                eval_result.bert_score_f1 = float(F1.mean())
            except Exception as e:
                print(f'  [BERTScore スキップ] {e}')

        return eval_result.compute_composite()

print('✅ 評価器定義完了')

## セル5: セマンティックキャッシュ（時間減衰付き）

In [ ]:
def get_tau(prompt: str) -> int:
    if any(kw in prompt for kw in ['ニュース', '速報', '今日', '最新', '現在']):
        return 300
    elif any(kw in prompt for kw in ['方法', '手順', '使い方']):
        return 21600
    elif any(kw in prompt for kw in ['とは', '定義', '概念', '原理', '歴史']):
        return 86400
    return 3600


def save_cache(prompt: str, response: str, emb: np.ndarray,
               best_method: str = 'none') -> None:
    tau = get_tau(prompt)
    key = f'cache:{len(r.keys("cache:*"))}'
    r.set(key, json.dumps({
        'prompt'     : prompt,
        'response'   : response,
        'embedding'  : emb.tolist(),
        'timestamp'  : time.time(),
        'length'     : token_len(response),
        'best_method': best_method,
        'tau'        : tau,
    }), ex=int(tau * 2))
    print(f'  [キャッシュ保存] key={key}, TTL={tau*2}s, 採用手法={best_method}')


def search_cache(prompt: str, threshold: float = 0.5) -> Optional[str]:
    now = time.time()
    tau = get_tau(prompt)
    query_emb = get_embedder().encode(prompt)

    print(f'  [キャッシュ検索] tau={tau}s, 閾値={threshold}')

    best_score, best_res = 0.0, None
    for key in r.keys('cache:*'):
        raw = r.get(key)
        if not raw:
            continue
        data = json.loads(raw)
        emb = np.array(data['embedding'])
        sim   = float(np.dot(query_emb, emb) /
                     (np.linalg.norm(query_emb) * np.linalg.norm(emb) + 1e-10))
        age   = now - data['timestamp']
        decay = float(np.exp(-age / tau))
        value = min(data['length'] / 100, 1.0)
        score = sim * decay * value
        print(f'    {key}: sim={sim:.3f} decay={decay:.3f} value={value:.3f} '
              f'→ score={score:.3f} [採用手法:{data.get("best_method","?")}]')
        if score > best_score:
            best_score = score
            best_res = data['response']

    if best_score >= threshold:
        print(f'  [キャッシュヒット ✅] best_score={best_score:.3f}')
        return best_res

    print(f'  [キャッシュミス ❌] best_score={best_score:.3f}')
    return None

print('✅ キャッシュシステム定義完了')

## セル6: メインパイプライン

**正しいフロー:**
1. キャッシュ検索
2. 全手法で圧縮
3. 全圧縮結果を評価
4. **総合スコア最高の手法を選択**
5. 選んだ圧縮後プロンプトでLLM呼び出し
6. キャッシュ保存

In [ ]:
# LLM呼び出し（本番ではOpenAI/AnthropicのAPIに差し替える）
_lm_tok = None
_lm_mdl = None

def call_llm(prompt: str) -> str:
    import torch
    global _lm_tok, _lm_mdl
    if _lm_mdl is None:
        print('  [モデルロード] flan-t5-base（LLM）...')
        _lm_tok = AutoTokenizer.from_pretrained('google/flan-t5-base')
        _lm_mdl = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
    print('  [LLM呼び出し 🤖]')
    inputs = _lm_tok(prompt, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = _lm_mdl.generate(**inputs, max_new_tokens=80)
    return _lm_tok.decode(outputs[0], skip_special_tokens=True)


def select_best_compression(
    prompt: str,
    compressor_names: list[str],
    compression_ratio: float,
    use_bert_score: bool = False,
) -> tuple[CompressionResult, EvaluationResult, list[EvaluationResult]]:
    """
    全手法で圧縮 → 評価 → 最良を返す。

    Returns:
        (最良のCompressionResult, 最良のEvaluationResult, 全評価結果リスト)
    """
    evaluator = CompressionEvaluator(use_bert_score=use_bert_score)
    all_evals: list[EvaluationResult] = []

    print(f'  [全手法で圧縮・評価 開始] 手法数={len(compressor_names)}')

    for name in compressor_names:
        try:
            compressor = get_compressor(name)
            comp_result = compressor.compress(prompt, ratio=compression_ratio)
            eval_result = evaluator.evaluate(comp_result)
            all_evals.append(eval_result)
            print(eval_result.summary_line())
        except Exception as e:
            print(f'  [{name}] ⚠️ スキップ: {e}')

    if not all_evals:
        raise RuntimeError('すべての圧縮手法が失敗しました')

    # 総合スコアで降順ソート
    all_evals.sort(key=lambda x: x.composite_score, reverse=True)
    best_eval = all_evals[0]
    best_comp = best_eval.compression_result

    print(f'  [最良手法を選択 ✨] {best_eval.method} '
          f'(総合スコア: {best_eval.composite_score:.3f})')

    return best_comp, best_eval, all_evals


def smart_llm(
    prompt: str,
    compressor_names: list[str] = None,
    compression_ratio: float = 0.3,
    cache_threshold: float = 0.5,
    use_bert_score: bool = False,
) -> dict:
    """
    正しいパイプライン:
      1. キャッシュ検索 → ヒットなら即返す
      2. 全手法で圧縮・評価
      3. 総合スコア最高の圧縮結果を採用
      4. 採用した圧縮後プロンプトでLLM呼び出し
      5. キャッシュ保存

    Args:
        prompt: 入力プロンプト
        compressor_names: 試す手法リスト（None=全手法）
        compression_ratio: 圧縮率（0〜1、削減する割合）
        cache_threshold: キャッシュヒット閾値
        use_bert_score: BERTScoreを評価に使う（重い）

    Returns:
        dict: response, cache_hit, best_method, all_evals, best_comp
    """
    if compressor_names is None:
        compressor_names = list(COMPRESSOR_REGISTRY.keys())

    print(f'\n{"█"*64}')
    print(f'  入力プロンプト: {prompt}')
    print(f'  試す手法: {compressor_names}')
    print(f'  圧縮率目標: {compression_ratio*100:.0f}%削減')
    print(f'{"─"*64}')

    # ─── Step 1: キャッシュ検索 ───────────────────────────────
    cached = search_cache(prompt, threshold=cache_threshold)
    if cached:
        return {
            'response'  : cached,
            'cache_hit' : True,
            'best_method': '(cached)',
            'all_evals' : [],
            'best_comp' : None,
        }

    # ─── Step 2 & 3: 全手法で圧縮 → 評価 → 最良を選択 ────────
    best_comp, best_eval, all_evals = select_best_compression(
        prompt, compressor_names, compression_ratio, use_bert_score
    )

    # ─── Step 4: LLM呼び出し（最良の圧縮後プロンプトで）─────────
    print(f'\n  [採用圧縮テキスト] {best_comp.compressed[:100]}...')
    response = call_llm(best_comp.compressed)

    # ─── Step 5: キャッシュ保存 ───────────────────────────────
    emb = get_embedder().encode(prompt)
    save_cache(prompt, response, emb, best_method=best_eval.method)

    return {
        'response'   : response,
        'cache_hit'  : False,
        'best_method': best_eval.method,
        'all_evals'  : all_evals,
        'best_comp'  : best_comp,
    }

print('✅ メインパイプライン定義完了')

## セル7: デモ実行

In [ ]:
# ============================================================
# デモ1: 基本動作確認
# 全手法で圧縮・評価 → 最良を選んでLLMに投げる
# ============================================================
print('デモ1: 基本動作確認（全手法評価 → 最良採用）')
print('='*64)

result = smart_llm(
    prompt='量子力学とは何ですか',
    compressor_names=['tfidf', 'selective', 'llmlingua', 'longlingua', 'attention'],
    compression_ratio=0.3,
    cache_threshold=0.5,
)

print(f'\n  採用手法  : {result["best_method"]}')
print(f'  LLM回答   : {result["response"]}')
print()
print('  ── 全手法スコア一覧 ──')
for ev in result['all_evals']:
    marker = ' ← 採用' if ev.method == result['best_method'] else ''
    print(ev.summary_line() + marker)

In [ ]:
# ============================================================
# デモ2: 2回目の同一クエリ → キャッシュヒット確認
# ============================================================
print('デモ2: 類似クエリ → キャッシュヒット確認')
print('='*64)

result2 = smart_llm(
    prompt='量子物理学の基本的な概念を教えて',  # 上と類似
    compression_ratio=0.3,
)

status = '💾 キャッシュ' if result2['cache_hit'] else f'🤖 LLM（採用手法: {result2["best_method"]}）'
print(f'\n  {status}')
print(f'  回答: {result2["response"]}')

In [ ]:
# ============================================================
# デモ3: 試す手法を絞り込む場合
# ============================================================
print('デモ3: 軽量手法のみで比較')
print('='*64)

result3 = smart_llm(
    prompt='最新のAIニュースを教えて',
    compressor_names=['tfidf', 'selective'],  # 軽量手法のみ
    compression_ratio=0.2,
)

print(f'\n  採用手法: {result3["best_method"]}')
print(f'  回答    : {result3["response"]}')

## セル8: カスタム手法の追加例

In [ ]:
# 独自手法を追加する場合は BaseCompressor を継承して
# COMPRESSOR_REGISTRY に1行追加するだけ

class LeadingNCompressor(BaseCompressor):
    """先頭N文を保持するシンプルな手法（ニュース記事など先頭重視の文書向け）"""
    @property
    def name(self): return 'LeadingN'

    def _compress(self, text: str, ratio: float) -> str:
        sentences = [s.strip() for s in re.split(r'[。.!?\n]', text) if s.strip()]
        keep_n = max(1, int(len(sentences) * (1 - ratio)))
        return '。'.join(sentences[:keep_n])

COMPRESSOR_REGISTRY['leadingn'] = LeadingNCompressor

print(f'手法追加後: {list(COMPRESSOR_REGISTRY.keys())}')

# 追加した手法も自動的に評価対象に含まれる
result4 = smart_llm(
    prompt='機械学習とは何ですか',
    compressor_names=['tfidf', 'leadingn'],  # 追加手法を含めて比較
    compression_ratio=0.3,
)
print(f'\n採用手法: {result4["best_method"]}')
print(f'回答    : {result4["response"]}')